In [ ]:
# %%
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

# %%
# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
VALIDATION_CSV = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Construction_Time_Estimate\validation_sample_50.csv")
PREDICTIONS_DIR = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Construction_Time_Estimate")
TIMESERIES_CSV = PREDICTIONS_DIR / 'all_ndas_ndvi_timeseries.csv'
OUTPUT_DIR = PREDICTIONS_DIR / 'Accuracy_Assessment'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_CSV_CANDIDATES = [
    PREDICTIONS_DIR / 'NHDA_construction_years_RF_v2.csv',
    PREDICTIONS_DIR / 'NHDA_construction_years_RF.csv',
]
PREDICTIONS_CSV = next((path for path in PREDICTIONS_CSV_CANDIDATES if path.exists()), PREDICTIONS_CSV_CANDIDATES[0])

DEFAULT_FIRST_IMAGE_YEAR = 2015
FALLBACK_SECOND_IMAGE_YEAR = 2016

# %%
# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------
def read_csv_flexible(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    for sep in [';', ',']:
        try:
            df = pd.read_csv(path, sep=sep, encoding='utf-8-sig')
            if df.shape[1] > 1:
                return df
        except Exception:
            pass
    raise ValueError(f"Could not parse CSV: {path}")


def normalize_id_col(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    if 'nhda_id' in df.columns:
        id_col = 'nhda_id'
    elif 'nda_id' in df.columns:
        id_col = 'nda_id'
    else:
        raise KeyError('No ID column found (expected nhda_id or nda_id).')
    out = df.rename(columns={id_col: 'nhda_id'}).copy()
    out['nhda_id'] = out['nhda_id'].astype(str).str.strip()
    return out


def build_validation_label(df_val: pd.DataFrame) -> pd.DataFrame:
    out = df_val.copy()
    if 'construction_start_year' not in out.columns:
        out['construction_start_year'] = np.nan
    if 'comment_start' not in out.columns:
        out['comment_start'] = ''
    out['comment_start'] = out['comment_start'].fillna('').astype(str).str.strip().str.lower()
    out['target_year_num'] = pd.to_numeric(out['construction_start_year'], errors='coerce')
    out['target_label'] = np.where(
        out['comment_start'].eq('already_built_up'),
        'already_built_up',
        np.where(out['target_year_num'].notna(), out['target_year_num'].astype('Int64').astype(str), np.nan)
    )
    return out


def build_first_image_year_map(df_ts: pd.DataFrame) -> pd.Series:
    ts = normalize_id_col(df_ts)
    if 'year' not in ts.columns or 'median' not in ts.columns:
        raise KeyError('Time series file must contain nhda_id/nda_id, year and median columns.')
    ts['year'] = pd.to_numeric(ts['year'], errors='coerce')
    ts['median'] = pd.to_numeric(ts['median'], errors='coerce')

    valid_2015 = (
        ts.loc[ts['year'].eq(DEFAULT_FIRST_IMAGE_YEAR) & ts['median'].notna(), ['nhda_id']]
        .drop_duplicates()
        .assign(first_image_year=DEFAULT_FIRST_IMAGE_YEAR)
    )

    first_year_map = pd.Series(FALLBACK_SECOND_IMAGE_YEAR, index=ts['nhda_id'].drop_duplicates().sort_values())
    if len(valid_2015) > 0:
        first_year_map.loc[valid_2015['nhda_id']] = DEFAULT_FIRST_IMAGE_YEAR
    first_year_map.name = 'first_image_year'
    return first_year_map


def normalize_prediction_label(df_pred: pd.DataFrame, first_image_year_map: pd.Series) -> pd.DataFrame:
    out = df_pred.copy()
    if 'construction_start_year' not in out.columns:
        raise KeyError('Predictions file must contain construction_start_year.')
    out = out.merge(first_image_year_map.rename('first_image_year'), on='nhda_id', how='left')
    out['first_image_year'] = out['first_image_year'].fillna(FALLBACK_SECOND_IMAGE_YEAR).astype(int)

    pred_raw = out['construction_start_year'].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan, '': np.nan})
    pred_num = pd.to_numeric(pred_raw, errors='coerce')
    out['pred_year_num'] = pred_num
    out['pred_is_first_year'] = pred_num.eq(out['first_image_year'])
    out['pred_label_raw'] = np.where(
        pred_raw.eq('already_built_up'),
        'already_built_up',
        np.where(pred_num.notna(), pred_num.astype('Int64').astype(str), np.nan)
    )
    out['pred_label'] = np.where(
        out['pred_label_raw'].eq('already_built_up') | out['pred_is_first_year'],
        'already_built_up',
        out['pred_label_raw']
    )
    return out


def fmt_pct(value, decimals=1):
    """Format a 0-1 float as a percentage string."""
    if pd.isna(value):
        return 'N/A'
    return f"{value * 100:.{decimals}f}%"


def fmt_n(value):
    return 'N/A' if pd.isna(value) else str(int(value))

# %%
# ------------------------------------------------------------------
# Load + normalize
# ------------------------------------------------------------------
val_raw = read_csv_flexible(VALIDATION_CSV)
pred_raw = read_csv_flexible(PREDICTIONS_CSV)
ts_raw = read_csv_flexible(TIMESERIES_CSV)

val = normalize_id_col(val_raw)
pred = normalize_id_col(pred_raw)

first_image_year_map = build_first_image_year_map(ts_raw)

val = build_validation_label(val)
pred = normalize_prediction_label(pred, first_image_year_map)

pred = pred.drop_duplicates(subset=['nhda_id'], keep='last').copy()

eval_df = val.merge(
    pred[[
        'nhda_id', 'construction_start_year', 'pred_year_num', 'pred_label_raw',
        'pred_label', 'pred_is_first_year', 'first_image_year', 'rf_confidence', 'method'
    ]],
    on='nhda_id',
    how='left',
    suffixes=('_val', '_pred')
)

print(f"Validation rows : {len(val)}")
print(f"Prediction rows : {len(pred)}")
print(f"Merged rows     : {len(eval_df)}")
print(f"Predictions CSV : {PREDICTIONS_CSV.name}")
print(f"Time series CSV : {TIMESERIES_CSV.name}")
print(f"NHDAs with first image year 2015: {(first_image_year_map == 2015).sum()}")
print(f"NHDAs with first image year 2016: {(first_image_year_map == 2016).sum()}")

# %%
# ------------------------------------------------------------------
# Core accuracy metrics
# ------------------------------------------------------------------
labeled = eval_df[eval_df['target_label'].notna()].copy()

N = len(labeled)
n_pred = labeled['pred_label'].notna().sum()
n_no_pred = N - n_pred
coverage = n_pred / N if N else np.nan
n_first_year_proxy = labeled['pred_is_first_year'].fillna(False).sum()

# Exact match (all classes; using per-NHDA first-image-year proxy for already_built_up)
labeled['exact_match'] = labeled['pred_label'] == labeled['target_label']
overall_accuracy = labeled['exact_match'].mean() if N else np.nan

# ── Numeric year subset ──────────────────────────────────────────
num = labeled[labeled['target_year_num'].notna()].copy()
num['abs_error'] = (num['pred_year_num'] - num['target_year_num']).abs()

N_num = len(num)
n_num_pred = num['pred_year_num'].notna().sum()
num_exact = (num['abs_error'] == 0).mean() if N_num else np.nan
num_pm1 = (num['abs_error'] <= 1).mean() if N_num else np.nan
num_pm2 = (num['abs_error'] <= 2).mean() if N_num else np.nan
num_pm5 = (num['abs_error'] <= 5).mean() if N_num else np.nan
mae = num['abs_error'].mean() if n_num_pred else np.nan
rmse = np.sqrt((num['abs_error'] ** 2).mean()) if n_num_pred else np.nan
median_ae = num['abs_error'].median() if n_num_pred else np.nan

# ── already_built_up subset ──────────────────────────────────────
abu = labeled[labeled['target_label'] == 'already_built_up'].copy()
abu_N = len(abu)
abu_recall = (abu['pred_label'] == 'already_built_up').mean() if abu_N else np.nan

# Precision for already_built_up
pred_abu = labeled[labeled['pred_label'] == 'already_built_up']
abu_precision = (pred_abu['target_label'] == 'already_built_up').mean() if len(pred_abu) else np.nan

abu_f1 = (
    2 * abu_precision * abu_recall / (abu_precision + abu_recall)
    if (abu_precision + abu_recall) > 0 else np.nan
)

# %%
# ------------------------------------------------------------------
# Print accuracy report
# ------------------------------------------------------------------
sep = "─" * 52

print(f"\n{'═'*52}")
print("  ACCURACY ASSESSMENT REPORT")
print(f"{'═'*52}")

print("\nOVERVIEW")
print(sep)
print(f"  Labeled validation samples      : {fmt_n(N):>8}")
print(f"  Samples with a prediction       : {fmt_n(n_pred):>8}  ({fmt_pct(coverage)})")
print(f"  Samples without prediction      : {fmt_n(n_no_pred):>8}  ({fmt_pct(1-coverage if coverage else np.nan)})")
print(f"  First-year proxy recodings      : {fmt_n(n_first_year_proxy):>8}")

print("\nOVERALL ACCURACY (all classes)")
print(sep)
print("  Based on adjusted prediction labels, where")
print("  construction_start_year equal to the first valid image year for that NHDA")
print("  is treated as already_built_up.")
print(f"  Overall Accuracy                : {fmt_pct(overall_accuracy):>8}  ({int(labeled['exact_match'].sum())}/{N} correct)")

print("\nNUMERIC YEAR ACCURACY")
print(sep)
print(f"  Samples                         : {fmt_n(N_num):>8}")
print(f"  Exact year match                : {fmt_pct(num_exact):>8}  ({int((num['abs_error'] == 0).sum())}/{N_num})")
print(f"  Within ±1 year                  : {fmt_pct(num_pm1):>8}  ({int((num['abs_error'] <= 1).sum())}/{N_num})")
print(f"  Within ±2 years                 : {fmt_pct(num_pm2):>8}  ({int((num['abs_error'] <= 2).sum())}/{N_num})")
print(f"  Within ±5 years                 : {fmt_pct(num_pm5):>8}  ({int((num['abs_error'] <= 5).sum())}/{N_num})")
print(f"  MAE  (mean abs. error)          : {mae:>7.2f} yr" if not pd.isna(mae) else "  MAE                             :      N/A")
print(f"  RMSE                            : {rmse:>7.2f} yr" if not pd.isna(rmse) else "  RMSE                            :      N/A")
print(f"  Median absolute error           : {median_ae:>7.2f} yr" if not pd.isna(median_ae) else "  Median AE                       :      N/A")

print("\nALREADY_BUILT_UP CLASS")
print(sep)
print(f"  Samples                         : {fmt_n(abu_N):>8}")
print(f"  Recall  (sensitivity)           : {fmt_pct(abu_recall):>8}  ({int((abu['pred_label'] == 'already_built_up').sum())}/{abu_N})")
print(f"  Precision                       : {fmt_pct(abu_precision):>8}")
print(f"  F1 Score                        : {fmt_pct(abu_f1):>8}")

print(f"\n{'═'*52}\n")

# %%
# ------------------------------------------------------------------
# Confusion matrix (crosstab)
# ------------------------------------------------------------------
conf = pd.crosstab(
    labeled['target_label'].fillna('NA'),
    labeled['pred_label'].fillna('no_prediction'),
    margins=True,
    margins_name='TOTAL'
)
print("CONFUSION MATRIX (rows = truth, cols = adjusted prediction)")
print(sep)
print(conf.to_string())
print()

# %%
# ------------------------------------------------------------------
# Error distribution table (numeric only)
# ------------------------------------------------------------------
bins = [0, 1, 2, 5, 10, np.inf]
labels = ['0 yr', '1 yr', '2 yr', '5 yr', '>5 yr']
num['error_bin'] = pd.cut(num['abs_error'], bins=bins, labels=labels, right=True)

err_dist = (
    num['error_bin']
    .value_counts()
    .reindex(labels)
    .reset_index()
)
err_dist.columns = ['max_abs_error', 'count']
err_dist['pct'] = err_dist['count'] / N_num * 100

print("ERROR DISTRIBUTION (numeric year predictions)")
print(sep)
print(f"{'Max Error':<12}  {'Count':>6}  {'Pct':>7}")
print("─" * 30)
for _, r in err_dist.iterrows():
    print(f"{r['max_abs_error']:<12}  {int(r['count']):>6}  {r['pct']:>6.1f}%")
print()

# %%
# ------------------------------------------------------------------
# Top-20 worst errors
# ------------------------------------------------------------------
start_col = 'construction_start_year_val' if 'construction_start_year_val' in labeled.columns else 'construction_start_year'

worst = (
    labeled
    .assign(abs_error=(labeled['pred_year_num'] - labeled['target_year_num']).abs())
    .sort_values(['abs_error', 'nhda_id'], ascending=[False, True])
    [[
        'nhda_id', start_col, 'comment_start', 'target_label', 'pred_label_raw',
        'pred_label', 'pred_year_num', 'pred_is_first_year', 'first_image_year',
        'rf_confidence', 'method', 'abs_error'
    ]]
    .head(20)
)
print("TOP-20 WORST PREDICTIONS")
print(sep)
print(worst.to_string(index=False))
print()

# %%
# ------------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------------
summary_rows = [
    ('labeled_samples', N, None),
    ('samples_with_prediction', n_pred, fmt_pct(coverage)),
    ('samples_without_prediction', n_no_pred, fmt_pct(1-coverage if coverage else np.nan)),
    ('first_year_proxy_recodings', int(n_first_year_proxy), None),
    ('overall_accuracy', int(labeled['exact_match'].sum()), fmt_pct(overall_accuracy)),
    ('numeric_samples', N_num, None),
    ('numeric_exact_match', int((num['abs_error'] == 0).sum()), fmt_pct(num_exact)),
    ('numeric_within_pm1_year', int((num['abs_error'] <= 1).sum()), fmt_pct(num_pm1)),
    ('numeric_within_pm2_years', int((num['abs_error'] <= 2).sum()), fmt_pct(num_pm2)),
    ('numeric_within_pm5_years', int((num['abs_error'] <= 5).sum()), fmt_pct(num_pm5)),
    ('numeric_mae_years', round(mae, 3) if not pd.isna(mae) else np.nan, None),
    ('numeric_rmse_years', round(rmse, 3) if not pd.isna(rmse) else np.nan, None),
    ('numeric_median_ae_years', round(median_ae, 3) if not pd.isna(median_ae) else np.nan, None),
    ('already_built_up_samples', abu_N, None),
    ('already_built_up_recall', None, fmt_pct(abu_recall)),
    ('already_built_up_precision', None, fmt_pct(abu_precision)),
    ('already_built_up_f1', None, fmt_pct(abu_f1)),
]

summary_df = pd.DataFrame(summary_rows, columns=['metric', 'count', 'pct'])

summary_path = OUTPUT_DIR / 'accuracy_assessment_summary.csv'
conf_path = OUTPUT_DIR / 'accuracy_assessment_confusion_matrix.csv'
merged_path = OUTPUT_DIR / 'accuracy_assessment_full_eval.csv'
errdist_path = OUTPUT_DIR / 'accuracy_assessment_error_distribution.csv'

summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
conf.to_csv(conf_path, encoding='utf-8-sig')
eval_df.to_csv(merged_path, index=False, encoding='utf-8-sig')
err_dist.to_csv(errdist_path, index=False, encoding='utf-8-sig')

print(f"Saved: {summary_path.name}")
print(f"Saved: {conf_path.name}")
print(f"Saved: {merged_path.name}")
print(f"Saved: {errdist_path.name}")